# Midsem Question 2 — Wordpiece Tokenization (2 merges)

**Corpus:** `low lower low lower lowest low lowest low lowest low new new new newer newer new`

**Task:** Apply the wordpiece algorithm for 2 merges. What is the tokenization of `newest`?

Wordpiece steps:
1. Split the corpus into words; split each word into characters, prefixing every non-initial character with `##`.
2. Count all adjacent symbol pairs across the corpus.
3. Merge the most frequent pair into a new symbol (add to vocabulary).
4. Repeat for 2 merges.
5. Tokenize `newest` using the learned merges.

In [1]:
from collections import Counter

CORPUS = "low lower low lower lowest low lowest low lowest low new new new newer newer new"
TARGET = "newest"
NUM_MERGES = 2


def word_to_symbols(word):
    """Split word into chars, prefix non-first chars with '##'."""
    if not word:
        return []
    syms = [word[0]]
    for ch in word[1:]:
        syms.append("##" + ch)
    return syms


def count_pairs(corpus_symbols):
    """Count adjacent pairs in a list of symbol-sequences."""
    c = Counter()
    for seq in corpus_symbols:
        for i in range(len(seq) - 1):
            c[(seq[i], seq[i + 1])] += 1
    return c


def apply_merge(corpus_symbols, pair):
    """Merge the given pair everywhere it appears; return new sequences."""
    a, b = pair
    merged = a + b
    out = []
    for seq in corpus_symbols:
        new_seq = []
        i = 0
        while i < len(seq):
            if i < len(seq) - 1 and seq[i] == a and seq[i + 1] == b:
                new_seq.append(merged)
                i += 2
            else:
                new_seq.append(seq[i])
                i += 1
        out.append(new_seq)
    return out


                                  
words = CORPUS.split()
seqs = [word_to_symbols(w) for w in words]

print("Initial word splits:")
for w, s in zip(words, seqs):
    print(f"  {w:8s} -> {s}")
print()

vocab = set()
for s in seqs:
    vocab.update(s)
print("Initial vocabulary:", sorted(vocab))
print()

merges = []
for it in range(1, NUM_MERGES + 1):
    pc = count_pairs(seqs)
    max_count = max(pc.values())
    top = [p for p, v in pc.items() if v == max_count]
    top.sort()
    best = top[0]
    merges.append((best, max_count))
    print(f"Merge {it}: pair={best}  count={max_count}")
    seqs = apply_merge(seqs, best)
    print("  Sequences after merge", it, ":")
    for w, s in zip(words, seqs):
        print(f"    {w:8s} -> {s}")
    print()

print("Learned merges:")
for i, (pair, cnt) in enumerate(merges, 1):
    print(f"  {i}. {pair} (count {cnt})")

                         
target_seq = word_to_symbols(TARGET)
for pair, _ in merges:
    target_seq = apply_merge([target_seq], pair)[0]

print()
print(f"Tokenization of '{TARGET}': {target_seq}")
print(f"Joined: {' '.join(target_seq)}")

Initial word splits:
  low      -> ['l', '##o', '##w']
  lower    -> ['l', '##o', '##w', '##e', '##r']
  low      -> ['l', '##o', '##w']
  lower    -> ['l', '##o', '##w', '##e', '##r']
  lowest   -> ['l', '##o', '##w', '##e', '##s', '##t']
  low      -> ['l', '##o', '##w']
  lowest   -> ['l', '##o', '##w', '##e', '##s', '##t']
  low      -> ['l', '##o', '##w']
  lowest   -> ['l', '##o', '##w', '##e', '##s', '##t']
  low      -> ['l', '##o', '##w']
  new      -> ['n', '##e', '##w']
  new      -> ['n', '##e', '##w']
  new      -> ['n', '##e', '##w']
  newer    -> ['n', '##e', '##w', '##e', '##r']
  newer    -> ['n', '##e', '##w', '##e', '##r']
  new      -> ['n', '##e', '##w']

Initial vocabulary: ['##e', '##o', '##r', '##s', '##t', '##w', 'l', 'n']

Merge 1: pair=('##o', '##w')  count=10
  Sequences after merge 1 :
    low      -> ['l', '##o##w']
    lower    -> ['l', '##o##w', '##e', '##r']
    low      -> ['l', '##o##w']
    lower    -> ['l', '##o##w', '##e', '##r']
    lowest   -> ['

## Answer

After 2 wordpiece merges, the most frequent pairs are drawn from the `low`/`lower`/`lowest` family (count 10 each), so the learned merges involve `l`/`o` symbols. The word **`newest`** contains none of those symbols, so no merge applies to it.

**Tokenization of `newest`:**

```
    n  e##  w##  e##  s##  t##
```

i.e. `["n", "e##", "w##", "e##", "s##", "t##"]` — the word stays fully split into characters because the two merges learned from the corpus do not cover any subword of `newest`.